In [3]:
import os, random
import numpy as np
import pandas as pd
from sklearn.cluster import OPTICS
from sklearn.datasets import make_blobs
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

INPUT_CSV = "/Users/suyashmali/Thesis/LLM/Notebook/Cleaned_Data_New1.csv"
TEXT_COL = "metadata_clean"
LABEL_COL = "Level3"

MIN_SAMPLES = 100
MAX_SAMPLES = 500

# EMBED_MODEL = "all-mpnet-base-v2"
EMBED_MODEL = "all-MiniLM-L6-v2"

In [4]:
df = pd.read_csv(INPUT_CSV)

def min_max_sample(df, label_col, min_k, max_k, seed=42):
    sampled_groups = []
    for label, g in df.groupby(label_col):
        if len(g) < min_k:
            continue
        n = min(len(g), max_k)
        sampled_groups.append(g.sample(n=n, random_state=seed))
    return pd.concat(sampled_groups).reset_index(drop=True)

sample_df = min_max_sample(df, LABEL_COL, MIN_SAMPLES, MAX_SAMPLES)

print("Final sample size:", len(sample_df))
print("Unique Level3 categories:", sample_df[LABEL_COL].nunique())

print(sample_df[[LABEL_COL]].value_counts().head(25))

Final sample size: 43803
Unique Level3 categories: 126
Level3                         
Wireless Routers                   500
Power Banks                        500
Network Switches                   500
Projector Mounts                   500
Projector Lamps                    500
Household Batteries                500
IT Support Services                500
Projector Accessories              500
Projection Screens                 500
Power Supply Units                 500
Power Distribution Units (PDUs)    500
Power Cables                       500
KVM Switches                       500
Keyboards                          500
Power Adapters & Inverters         500
Flat Panel Spare Parts             500
Portable Speakers                  500
PCs/Workstations                   500
Mice                               500
Mobile Device Chargers             500
PC/Workstation Barebones           500
Mobile Device Keyboards            500
Mobile Headsets                    500
Notebooks       

In [5]:
sample_df.columns

Index(['Brand', 'BrandPartCode', 'ProductName', 'Description.LongProductName',
       'Description.LongDesc', 'SummaryDescription.LongSummaryDescription',
       'pathlist_names', 'Level1', 'Level2', 'Level3', 'metadata_text',
       'metadata_clean'],
      dtype='object')

In [7]:
def coverage_by_min_k(df, min_k, level3="Level3", level2="Level2", level1="Level1"):
    # keep only Level3 categories with >= min_k products
    filtered = df.groupby(level3).filter(lambda x: len(x) >= min_k)

    return {
        "min_products": min_k,
        "total_products": len(filtered),
        "unique_Level3": filtered[level3].nunique(),
        "unique_Level2": filtered[level2].nunique(),
        "unique_Level1": filtered[level1].nunique(),
    }


min_values = [0, 20, 40, 45, 50, 70, 100, 150]

coverage_table = pd.DataFrame(
    [coverage_by_min_k(df, k) for k in min_values]
)

coverage_table

,min_products,total_products,unique_Level3,unique_Level2,unique_Level1
0,0,289865,209,14,1
1,20,289725,201,14,1
2,40,288693,165,14,1
3,45,288480,160,14,1
4,50,288247,155,13,1
5,70,287483,142,13,1
6,100,286118,126,13,1
7,150,282935,100,13,1


In [8]:
def dropped_categories(df, min_k, level3="Level3"):
    valid = set(
        df.groupby(level3)
        .filter(lambda x: len(x) >= min_k)[level3]
    )
    all_cats = set(df[level3])
    return all_cats - valid

dropped_categories(df, 100)

{'AV Conferencing Bridges',
 'AV Receivers',
 'Activity Trackers',
 'All-in-One PC/Workstation Mounts & Stands',
 'Audio Turntables',
 'Backpack PCs',
 'Bridges & Repeaters',
 'CPU Holders',
 'CRT TVs',
 'Cable Boots',
 'Cable Protectors',
 'Cable Splitters or Combiners',
 'Car Kits',
 'Cassette Players',
 'Computer TV Tuners',
 'Customer Displays',
 'E-Book Readers',
 'Fax Supplies',
 'Fibre Optic Adapters',
 'FireWire Cables',
 'Gateways/Controllers',
 'Handheld Mobile Computer Spare Parts',
 'Handheld Mobile Computers',
 'Head-Mounted Displays',
 'ISDN Access Devices',
 'IT Courses',
 'KVM Extenders',
 "Kids' Tablet Accessories",
 "Kids' Tablets",
 'Lamination Films',
 'Laminator Pouches',
 'MP3/MP4 Player Accessories',
 'Mobile Device Skins',
 'Modems',
 'Monitors CRT',
 'Network Cable Testers',
 'Network Extenders',
 'Network Management Devices',
 'Network Splitters',
 'Network Switch Modules',
 'Notebook Power Tips',
 'Numeric Keypads',
 'Other Input Devices',
 'PS/2 Cables',
 'P

In [9]:
print(sample_df[[LABEL_COL]].value_counts().head(130))

Level3                        
Wireless Routers                  500
Power Banks                       500
Network Switches                  500
Projector Mounts                  500
Projector Lamps                   500
                                 ... 
Portable Stereo Systems           108
Composite Video Cables            106
Component (YPbPr) Video Cables    104
Mobile Phone Cables               103
Input Device Accessories          100
Name: count, Length: 126, dtype: int64


In [10]:
sample_df["final_text"] = (
    sample_df[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.strip()
)

fallback_cols = ["ProductName", "Title"]

def fallback_text(row):
    if row["final_text"]:
        return row["final_text"]
    return " ".join(
        str(row[c]) for c in fallback_cols
        if c in row and pd.notna(row[c])
    )

sample_df["final_text"] = sample_df.apply(fallback_text, axis=1)

print("Non-empty texts:", sample_df["final_text"].astype(bool).sum())

Non-empty texts: 43803


In [11]:
sample_df["final_text"].head(2)

0    vertiv ems1000t, avocent. type av transmitter,...
1    c2g 29225. type av transmitter, maximum resolu...
Name: final_text, dtype: object

In [12]:
sample_df["metadata_clean"].head(2)

0    vertiv ems1000t, avocent. type av transmitter,...
1    c2g 29225. type av transmitter, maximum resolu...
Name: metadata_clean, dtype: object

In [13]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("max_colwidth", None)

In [26]:
model = SentenceTransformer(EMBED_MODEL)

embeddings = model.encode(
    sample_df["metadata_clean"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/343 [00:00<?, ?it/s]

Embedding shape: (43803, 384)


In [29]:
umap_model = UMAP(
    n_neighbors=15,
    min_dist=0.0,
    n_components=25,
    metric="cosine",
    random_state=RANDOM_STATE
)

reduced = umap_model.fit_transform(embeddings)
print("Reduced shape:", reduced.shape)

/opt/anaconda3/envs/taxonomy_env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Reduced shape: (43803, 25)


In [28]:
optics = OPTICS(
    min_samples=15,         
    metric="euclidean",
    cluster_method="xi",    
    xi=0.05,              
    min_cluster_size = 100
)

sample_df["C_id_optics"] = optics.fit_predict(reduced)

print("Clusters formed:", sample_df["C_id_optics"].nunique())
print(sample_df["C_id_optics"].value_counts().head(10))

Clusters formed: 179
C_id_optics
-1      20372
 64       361
 171      300
 168      220
 3        214
 146      205
 49       202
 61       195
 91       190
 11       185
Name: count, dtype: int64


In [30]:
ari = adjusted_rand_score(sample_df[LABEL_COL], sample_df["C_id_optics"])
nmi = normalized_mutual_info_score(sample_df[LABEL_COL], sample_df["C_id_optics"])

print("ARI:", round(ari, 4))
print("NMI:", round(nmi, 4))

ARI: 0.0121
NMI: 0.53


In [18]:
def cluster_purity(df, cluster_col, gold_col):
    stats = (
        df.groupby(cluster_col)[gold_col]
        .value_counts()
        .rename("count")
        .reset_index()
    )
    top = stats.groupby(cluster_col).first().reset_index()
    sizes = df[cluster_col].value_counts().rename("size").reset_index()
    merged = top.merge(sizes, on=cluster_col)
    merged["purity"] = merged["count"] / merged["size"]
    return merged.sort_values("purity", ascending=False)

purity_df = cluster_purity(sample_df, "C_id_optics", LABEL_COL)
purity_df.head(100)

,C_id_optics,Level3,count,size,purity
39,38,Household Batteries,116,116,1.000000
46,45,Power Distribution Units (PDUs),133,133,1.000000
142,141,Projector Lamps,103,103,1.000000
51,50,Network Transceiver Modules,101,101,1.000000
73,72,Mobile Headsets,106,106,1.000000
176,175,Calculators,121,121,1.000000
170,169,Flat Panel Spare Parts,105,105,1.000000
152,151,Thin Clients,139,139,1.000000
80,79,MP3/MP4 Players,107,107,1.000000
49,48,Power Adapters & Inverters,157,158,0.993671


In [36]:
example_cluster = sample_df["C_id_optics"].value_counts().index[4]

sample_df[sample_df["C_id_optics"] == example_cluster][["Brand", "ProductName", LABEL_COL]].head(100)

,Brand,ProductName,Level3
1271,APC,RBC59,Battery Chargers
1288,Approx,APPBC02,Battery Chargers
1371,Approx,APPBC01,Battery Chargers
1404,APC,Battery Management Harness,Battery Chargers
10271,APC,WBATTREPLC-G3-00,Installation Services
10352,APC,WBATTREPLC-SL-00,Installation Services
20129,APC,SBP10KHC3M1,Power Supply Units
20174,APC,SBP10KHR2M1,Power Supply Units
20197,APC,SYMMETRA BATTERY FRAME,Power Supply Units
20340,APC,SBP10KHR3M1,Power Supply Units


In [34]:
example_cluster = sample_df["C_id_optics"].value_counts().index[14]

sample_df[sample_df["C_id_optics"] == example_cluster][["Brand", "ProductName", LABEL_COL]].head(100)

,Brand,ProductName,Level3
7399,Samsung,WMN300SB,Flat Panel Wall Mounts
25668,Samsung,HW-M550/ZA,Soundbar Speakers
25675,Samsung,HW-H751,Soundbar Speakers
25678,Samsung,HW-J6501R,Soundbar Speakers
25683,Samsung,HW-K651/EN,Soundbar Speakers
25688,Samsung,HW-H551,Soundbar Speakers
25694,Samsung,HW-H7501,Soundbar Speakers
25697,Samsung,HW-J8501,Soundbar Speakers
25704,Sharp,HT-SL70,Soundbar Speakers
25708,Samsung,HW-J8501R,Soundbar Speakers


In [45]:
def cluster_quality(df, cluster_col, gold_col):
    stats = (
        df.groupby(cluster_col)[gold_col]
        .value_counts()
        .rename("count")
        .reset_index()
    )

    sizes = df[cluster_col].value_counts().rename("size").reset_index()
    merged = stats.merge(sizes, on=cluster_col)

    # purity
    top = (
        merged.sort_values("count", ascending=False)
        .groupby(cluster_col)
        .first()
        .reset_index()
    )
    top["purity"] = top["count"] / top["size"]
    top["impurity"] = 1 - top["purity"]

    # entropy
    entropy = (
        merged.assign(p=lambda x: x["count"] / x["size"])
        .groupby(cluster_col)
        .apply(lambda g: -np.sum(g["p"] * np.log(g["p"])))
        .reset_index(name="entropy")
    )

    return top.merge(entropy, on=cluster_col).sort_values("impurity", ascending=False)


In [46]:
quality_df = cluster_quality(sample_df, "C_id_agg", LABEL_COL)

# Worst clusters by impurity
quality_df.head(1000)

/var/folders/b_/cv34fjz13m34nzs9_x_gpp600000gn/T/ipykernel_50831/315832772.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: -np.sum(g["p"] * np.log(g["p"])))


,C_id_agg,Level3,count,size,purity,impurity,entropy
103,103,Keyboards,1,8,0.125000,0.875000,2.079442
133,133,Wire Connectors,1,7,0.142857,0.857143,1.945910
170,170,VGA Cables,268,1859,0.144164,0.855836,2.384614
39,39,DVD/Blu-Ray Players,298,1504,0.198138,0.801862,1.972885
76,76,Signage Displays,300,1494,0.200803,0.799197,1.635369
45,45,Flat Panel Desk Mounts,282,1304,0.216258,0.783742,1.823586
126,126,All-in-One PCs/Workstations,294,1335,0.220225,0.779775,1.752583
58,58,Server Barebones,300,1330,0.225564,0.774436,1.763303
5,5,Soundbar Speakers,298,1299,0.229407,0.770593,1.795150
42,42,Data Projectors,299,1283,0.233048,0.766952,1.713945


In [29]:
example_cluster = sample_df["C_id_agg"].value_counts().index[49]

sample_df[sample_df["C_id_agg"] == example_cluster][["Brand", "ProductName", LABEL_COL]].head(1000)

,Brand,ProductName,Level3
5697,Samsung,MID-UD55MB,Flat Panel Accessories
5703,Salora,Neck P32AT2BXHS7850,Flat Panel Accessories
5705,NEC,SP-4615,Flat Panel Accessories
5717,Salora,Stand Left P15ATGT16L00H,Flat Panel Accessories
5724,Salora,Stand P15ATBDAHG19400000H,Flat Panel Accessories
5732,Salora,Stand Right (P15ATBDAHG6665RD000H),Flat Panel Accessories
5738,NEC,SP-5220,Flat Panel Accessories
5742,NEC,SP-32,Flat Panel Accessories
5743,Samsung,MID40-UX3,Flat Panel Accessories
5745,Samsung,MID462-UT2,Flat Panel Accessories


In [35]:
sample_df[sample_df["C_id_"] == 49][["Brand", "ProductName", LABEL_COL]].head(1000)

KeyError: 'C_id_agg'

In [47]:
sample_df[sample_df["C_id_agg"] == 49][["Brand", "ProductName", LABEL_COL]].nunique()

Brand          3
ProductName    5
Level3         3
dtype: int64

In [60]:
sample_df[sample_df["C_id_agg"] == 200][["Brand", "ProductName", LABEL_COL]].nunique()

Brand            4
ProductName    267
Level3           1
dtype: int64

In [51]:
sample_df[sample_df["C_id_agg"] == 200][["Brand", "ProductName", LABEL_COL]].head(100)

,Brand,ProductName,Level3
1921,Casio,MS-20NC,Calculators
1922,Sharp,EL-243S,Calculators
1923,Canon,F-604,Calculators
1924,HP,12c,Calculators
1925,Sharp,EL-339HB,Calculators
1926,Casio,SL-100NC,Calculators
1927,Casio,JW-200TV-RD,Calculators
1928,Sharp,EL-531XGVL,Calculators
1929,Sharp,EL-334FB,Calculators
1930,HP,10bII+,Calculators


In [ ]:
sample_df["Level3"].head(2)